# 4. Hybrid: lemma chunks + calibration kNN

Основной классический submission: лексический поиск по чанкам объединяется с обновлённым calibration kNN. Точные дубли не удаляются.

In [1]:
!pip install pyarrow beautifulsoup4 lxml pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 671.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 22.6 MB/s eta 0:00:00


In [2]:
import sys
import subprocess
import importlib.util

In [3]:
from pathlib import Path
from functools import lru_cache
from collections import defaultdict, Counter
import html
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

pd.set_option('display.max_colwidth', 140)
SEED = 42
np.random.seed(SEED)

In [4]:
ARTICLE_PATH = Path('/kaggle/input/datasets/naruto2009/avito-bootcamp-dataset/candidate_public/candidate_data/articles.f')
CALIBRATION_PATH = Path('/kaggle/input/datasets/naruto2009/avito-bootcamp-dataset/candidate_public/candidate_data/calibration.f')
TEST_PATH = Path('/kaggle/input/datasets/naruto2009/avito-bootcamp-dataset/candidate_public/candidate_data/test.f')

articles = pd.read_feather(ARTICLE_PATH)
calibration = pd.read_feather(CALIBRATION_PATH)
test = pd.read_feather(TEST_PATH)

In [5]:
import pymorphy3

POLITE_PATTERNS = [
    r'\bздравствуйте\b', r'\bдобрый\s+(день|вечер|утро)\b',
    r'\bподскажите\b', r'\bпожалуйста\b', r'\bскажите\b',
    r'\bбудьте\s+добры\b', r'\bспасибо\b',
]
BLOCK_TAGS = ['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'li', 'th', 'td', 'label']


def normalize_text(text: str, remove_politeness: bool = False) -> str:
    text = html.unescape(str(text or '')).lower().replace('ё', 'е')
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    if remove_politeness:
        for pattern in POLITE_PATTERNS:
            text = re.sub(pattern, ' ', text)
    text = re.sub(r'[^0-9a-zа-я]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def html_to_text(raw_html: str) -> str:
    soup = BeautifulSoup(str(raw_html or ''), 'lxml')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    for image in soup.find_all('img'):
        alt = image.get('alt', '')
        if alt:
            image.replace_with(' ' + alt + ' ')
    return normalize_text(soup.get_text(' ', strip=True))


def html_to_blocks(raw_html: str) -> list[str]:
    soup = BeautifulSoup(str(raw_html or ''), 'lxml')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    for image in soup.find_all('img'):
        alt = image.get('alt', '')
        if alt:
            image.replace_with(' ' + alt + ' ')

    blocks = []
    for tag in soup.find_all(BLOCK_TAGS):
        text = normalize_text(tag.get_text(' ', strip=True))
        if text and (not blocks or text != blocks[-1]):
            blocks.append(text)
    if not blocks:
        fallback = normalize_text(soup.get_text(' ', strip=True))
        if fallback:
            blocks = [fallback]
    return blocks


def make_chunks(title: str, raw_html: str, max_chars: int = 1100, overlap_blocks: int = 1) -> list[str]:
    title_clean = normalize_text(title)
    blocks = html_to_blocks(raw_html)
    chunks, current = [], []
    current_len = 0
    for block in blocks:
        if current and current_len + len(block) + 1 > max_chars:
            chunk = normalize_text(title_clean + ' ' + ' '.join(current))
            if chunk:
                chunks.append(chunk)
            current = current[-overlap_blocks:] if overlap_blocks else []
            current_len = sum(len(x) + 1 for x in current)
        current.append(block)
        current_len += len(block) + 1
    if current:
        chunk = normalize_text(title_clean + ' ' + ' '.join(current))
        if chunk:
            chunks.append(chunk)
    return list(dict.fromkeys(chunks)) or [title_clean]


morph = pymorphy3.MorphAnalyzer()


@lru_cache(maxsize=200_000)
def lemmatize_token(token: str) -> str:
    token = token.lower().replace('ё', 'е')
    if token.isdigit() or len(token) <= 1:
        return token
    if not any('а' <= char <= 'я' for char in token):
        return token
    return morph.parse(token)[0].normal_form


def lemmatize_text(text: str) -> str:
    return ' '.join(lemmatize_token(token) for token in normalize_text(text).split())


def prepare_queries(series: pd.Series) -> list[str]:
    return [normalize_text(x, remove_politeness=True) for x in series.fillna('')]

In [6]:
def parse_ground_truth(value: str) -> list[int]:
    return [int(x) for x in str(value).split() if x.strip()]


def average_precision_at_k(relevant, predicted, k=10) -> float:
    relevant = set(relevant)
    if not relevant:
        return 0.0
    score = 0.0
    hits = 0
    used = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in relevant and article_id not in used:
            hits += 1
            score += hits / rank
            used.add(article_id)
    return score / min(len(relevant), k)


def map_at_k(ground_truths, predictions, k=10) -> float:
    return float(np.mean([
        average_precision_at_k(gt, pred, k=k)
        for gt, pred in zip(ground_truths, predictions)
    ]))


def recall_at_k(ground_truths, predictions, k=10) -> float:
    values = []
    for gt, pred in zip(ground_truths, predictions):
        gt = set(gt)
        values.append(len(gt & set(pred[:k])) / len(gt))
    return float(np.mean(values))


def scores_to_predictions(scores: np.ndarray, article_ids: np.ndarray, k: int = 10) -> list[list[int]]:
    order = np.argsort(-scores, axis=1)[:, :k]
    return [[int(article_ids[j]) for j in row] for row in order]


def reciprocal_rank_fusion(score_matrices, weights=None, rrf_k=60) -> np.ndarray:
    if weights is None:
        weights = [1.0] * len(score_matrices)
    fused = np.zeros_like(score_matrices[0], dtype=np.float32)
    n_queries, n_docs = fused.shape
    rows = np.arange(n_queries)[:, None]
    for scores, weight in zip(score_matrices, weights):
        order = np.argsort(-scores, axis=1)
        ranks = np.empty_like(order)
        ranks[rows, order] = np.arange(1, n_docs + 1)[None, :]
        fused += weight / (rrf_k + ranks)
    return fused


def validate_and_save_submission(test_df, predictions, articles_df, filename='/kaggle/working/answer.csv'):
    valid_ids = set(articles_df['article_id'].astype(int))
    answers = []
    for pred in predictions:
        deduplicated = list(dict.fromkeys(int(x) for x in pred))[:10]
        assert 1 <= len(deduplicated) <= 10
        assert set(deduplicated).issubset(valid_ids)
        answers.append(' '.join(map(str, deduplicated)))
    submission = pd.DataFrame({'query_id': test_df['query_id'].values, 'answer': answers})
    assert len(submission) == len(test_df)
    assert submission['query_id'].is_unique
    submission.to_csv(filename, index=False)
    print(f'Сохранено: {filename}, shape={submission.shape}')
    display(submission.head())
    return submission

In [7]:
def build_chunk_frame(articles_df, max_chars=1100, overlap_blocks=1):
    rows = []
    for article_pos, row in articles_df.reset_index(drop=True).iterrows():
        for chunk_pos, chunk in enumerate(make_chunks(row['title'], row['body'], max_chars, overlap_blocks)):
            rows.append({
                'article_pos': article_pos,
                'article_id': int(row['article_id']),
                'chunk_pos': chunk_pos,
                'chunk_text': chunk,
            })
    result = pd.DataFrame(rows)
    result['lemma_chunk_text'] = result['chunk_text'].map(lemmatize_text)
    return result


def aggregate_chunk_scores(chunk_scores, chunk_article_positions, n_articles, second_weight=0.30):
    result = np.zeros((chunk_scores.shape[0], n_articles), dtype=np.float32)
    groups = defaultdict(list)
    for chunk_idx, article_pos in enumerate(chunk_article_positions):
        groups[int(article_pos)].append(chunk_idx)
    for article_pos, chunk_indices in groups.items():
        values = chunk_scores[:, chunk_indices]
        if len(chunk_indices) == 1:
            result[:, article_pos] = values[:, 0]
        else:
            top_two = np.partition(values, -2, axis=1)[:, -2:]
            result[:, article_pos] = top_two.max(axis=1) + second_weight * top_two.min(axis=1)
    return result

In [8]:
def make_label_matrix(calibration_df, article_ids):
    article_to_pos = {int(article_id): pos for pos, article_id in enumerate(article_ids)}
    labels = np.zeros((len(calibration_df), len(article_ids)), dtype=np.float32)
    for query_pos, value in enumerate(calibration_df['ground_truth']):
        for article_id in parse_ground_truth(value):
            labels[query_pos, article_to_pos[article_id]] = 1.0
    return labels


def knn_vote_scores(similarities, label_matrix, k=50, exclude_self=False, frequency_penalty=0.15):
    similarities = similarities.copy()
    if exclude_self and similarities.shape[0] == similarities.shape[1]:
        np.fill_diagonal(similarities, -np.inf)
    available = similarities.shape[1] - (1 if exclude_self else 0)
    k = min(k, available)
    neighbor_indices = np.argpartition(-similarities, kth=k - 1, axis=1)[:, :k]
    result = np.zeros((similarities.shape[0], label_matrix.shape[1]), dtype=np.float32)
    frequency = np.maximum(label_matrix.sum(axis=0), 1.0) ** frequency_penalty
    for query_pos, indices in enumerate(neighbor_indices):
        weights = np.maximum(similarities[query_pos, indices], 0.0) ** 2
        if weights.sum() > 0:
            result[query_pos] = weights @ label_matrix[indices]
    return result / frequency[None, :]

In [9]:
article_ids = articles['article_id'].astype(int).to_numpy()
cal_queries = prepare_queries(calibration['query_text'])
test_queries = prepare_queries(test['query_text'])
cal_lemma_queries = [lemmatize_text(x) for x in cal_queries]
test_lemma_queries = [lemmatize_text(x) for x in test_queries]
truths = calibration['ground_truth'].map(parse_ground_truth).tolist()
label_matrix = make_label_matrix(calibration, article_ids)
chunks = build_chunk_frame(articles)
chunk_to_article = chunks['article_pos'].to_numpy()
print('Статей:', len(articles), 'чанков:', len(chunks))

Статей: 793 чанков: 5134


In [10]:
# Chunk retrieval: char + lemma word. Raw word оставлен дополнительным каналом.
raw_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), sublinear_tf=True, max_features=160_000,
    token_pattern=r'(?u)\b[0-9a-zа-я]{2,}\b'
)
raw_chunks = raw_vectorizer.fit_transform(chunks['chunk_text'])
raw_cal_chunks = (raw_vectorizer.transform(cal_queries) @ raw_chunks.T).toarray().astype(np.float32)
raw_test_chunks = (raw_vectorizer.transform(test_queries) @ raw_chunks.T).toarray().astype(np.float32)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True, min_df=2, max_features=220_000
)
char_chunks = char_vectorizer.fit_transform(chunks['chunk_text'])
char_cal_chunks = (char_vectorizer.transform(cal_queries) @ char_chunks.T).toarray().astype(np.float32)
char_test_chunks = (char_vectorizer.transform(test_queries) @ char_chunks.T).toarray().astype(np.float32)

lemma_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), sublinear_tf=True, max_features=160_000,
    token_pattern=r'(?u)\b[0-9a-zа-я]{2,}\b'
)
lemma_chunks = lemma_vectorizer.fit_transform(chunks['lemma_chunk_text'])
lemma_cal_chunks = (lemma_vectorizer.transform(cal_lemma_queries) @ lemma_chunks.T).toarray().astype(np.float32)
lemma_test_chunks = (lemma_vectorizer.transform(test_lemma_queries) @ lemma_chunks.T).toarray().astype(np.float32)

raw_cal = aggregate_chunk_scores(raw_cal_chunks, chunk_to_article, len(articles))
raw_test = aggregate_chunk_scores(raw_test_chunks, chunk_to_article, len(articles))
char_cal = aggregate_chunk_scores(char_cal_chunks, chunk_to_article, len(articles))
char_test = aggregate_chunk_scores(char_test_chunks, chunk_to_article, len(articles))
lemma_cal = aggregate_chunk_scores(lemma_cal_chunks, chunk_to_article, len(articles))
lemma_test = aggregate_chunk_scores(lemma_test_chunks, chunk_to_article, len(articles))

chunk_cal_scores = reciprocal_rank_fusion([raw_cal, char_cal, lemma_cal], weights=[0.5, 1.0, 1.0])
chunk_test_scores = reciprocal_rank_fusion([raw_test, char_test, lemma_test], weights=[0.5, 1.0, 1.0])

In [11]:
# Query-to-query similarity: raw + char + lemma.
q_raw_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), sublinear_tf=True, max_features=80_000,
    token_pattern=r'(?u)\b[0-9a-zа-я]{2,}\b'
)
cal_q_raw = q_raw_vectorizer.fit_transform(cal_queries)
test_q_raw = q_raw_vectorizer.transform(test_queries)

q_char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True, max_features=100_000
)
cal_q_char = q_char_vectorizer.fit_transform(cal_queries)
test_q_char = q_char_vectorizer.transform(test_queries)

q_lemma_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), sublinear_tf=True, max_features=80_000,
    token_pattern=r'(?u)\b[0-9a-zа-я]{2,}\b'
)
cal_q_lemma = q_lemma_vectorizer.fit_transform(cal_lemma_queries)
test_q_lemma = q_lemma_vectorizer.transform(test_lemma_queries)

cal_similarity = (
    0.20 * (cal_q_raw @ cal_q_raw.T).toarray()
    + 0.50 * (cal_q_char @ cal_q_char.T).toarray()
    + 0.30 * (cal_q_lemma @ cal_q_lemma.T).toarray()
).astype(np.float32)
test_similarity = (
    0.20 * (test_q_raw @ cal_q_raw.T).toarray()
    + 0.50 * (test_q_char @ cal_q_char.T).toarray()
    + 0.30 * (test_q_lemma @ cal_q_lemma.T).toarray()
).astype(np.float32)

In [12]:
rows = []
for k in [20, 30, 50]:
    knn_cal = knn_vote_scores(cal_similarity, label_matrix, k=k, exclude_self=True)
    for knn_weight in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0]:
        scores = reciprocal_rank_fusion([chunk_cal_scores, knn_cal], weights=[1.0, knn_weight])
        predictions = scores_to_predictions(scores, article_ids)
        rows.append({
            'k': k, 'knn_weight': knn_weight,
            'MAP@10_LOO': map_at_k(truths, predictions),
            'Recall@10_LOO': recall_at_k(truths, predictions),
        })
result = pd.DataFrame(rows).sort_values('MAP@10_LOO', ascending=False)
display(result.head(15))
BEST_K = int(result.iloc[0]['k'])
BEST_KNN_WEIGHT = float(result.iloc[0]['knn_weight'])

,k,knn_weight,MAP@10_LOO,Recall@10_LOO
17,50,4.0,0.627052,0.900000
11,30,4.0,0.624656,0.887667
16,50,3.0,0.621603,0.906667
10,30,3.0,0.619702,0.895333
15,50,2.5,0.617163,0.908667
14,50,2.0,0.615995,0.909333
5,20,4.0,0.613344,0.867333
9,30,2.5,0.612266,0.895667
4,20,3.0,0.611896,0.869333
3,20,2.5,0.610397,0.871333


In [13]:
knn_test_scores = knn_vote_scores(test_similarity, label_matrix, k=BEST_K, exclude_self=False)
final_test_scores = reciprocal_rank_fusion(
    [chunk_test_scores, knn_test_scores], weights=[1.0, BEST_KNN_WEIGHT]
)
test_predictions = scores_to_predictions(final_test_scores, article_ids)
submission = validate_and_save_submission(test, test_predictions, articles)

Сохранено: /kaggle/working/answer.csv, shape=(500, 2)


,query_id,answer
0,1,4234 4396 4308 4219 4214 4440 2646 4384 2698 1951
1,2,4219 4384 2646 4400 4440 4234 4308 1951 4387 1958
2,3,1909 4234 4396 4308 4219 4328 1907 4400 4286 4403
3,4,4219 4234 2646 1966 2865 4400 4214 4384 4361 2698
4,5,4214 2698 4308 4219 2948 4286 1951 4234 4328 2646
